# All computations (2D)

In [1]:
import numpy as np
import pandas as pd
import os

import sys
sys.path.append('..')

from surface_functions_2D import compute_surface_variance, compute_surface_roughness

## Dataframe computations

### 2. Roughness

In [2]:
summary_rows = []
for typenoise in ["TimeDep", "Columnar"]:
    
    for delta, deltaname in zip([0, np.arctan(5), np.atan(10)], ["0", "atan5", "atan10"]):

        path = f"../Simulations_2D/Output{typenoise}/all_simulations_delta{deltaname}.pkl"
        if not os.path.exists(path):
            print(f"File {path} not found, skipping.")
            continue

        print(f"## {typenoise} – Delta {deltaname}")

        # Load previously saved simulations
        df = pd.read_pickle(path)

        roughness_list = []  # used to compute per-sim roughness
        variance_list = []   # used to compute per-sim variance
        datapoint_list = []

        # Loop over simulations (one per seed / row)
        for _, row in df.iterrows():
            t = row["t"]
            phases = row["phases"]

            # ensure array, not list-of-lists
            variance = np.asarray(compute_surface_variance(phases), dtype=float)
            roughness = np.asarray(compute_surface_roughness(phases), dtype=float)

            variance_list.append(variance)
            roughness_list.append(roughness)
            datapoint_list.append(len(roughness))

        # Avoid chained-assignment issues
        df = df.copy()
        df["variance"] = variance_list
        df["roughness"] = roughness_list
        df["datapoints"] = datapoint_list

        # Save back the updated simulations DataFrame (with roughness column)
        df.to_pickle(path)

        ## AVERAGE ROUGHNESS PER (K, L, T) ##
        # group by K, L, T and compute mean roughness for each group
        for (K_val, L_val, T_val), group in df.groupby(["K", "L", "T"]):

            rough_arrays = [np.asarray(r, dtype=float) for r in group["roughness"]]
            mean_roughness = np.mean(np.stack(rough_arrays, axis=0), axis=0)

            variance_arrays = [np.asarray(r, dtype=float) for r in group["variance"]]
            mean_roughness2 = np.sqrt(np.mean(np.stack(variance_arrays, axis=0), axis=0))

            # assume all t are identical within group; take the first
            t = group.iloc[0]["t"]
            seed_list = group.index.tolist()
            M = len(seed_list) 

            avg_id = f"{typenoise}_delta{deltaname}_L{L_val}_K{K_val}_T{T_val}_M{M}"

            summary_rows.append(
                {
                    "avg_id": avg_id,
                    "typenoise": typenoise,
                    "delta": delta,
                    "deltaname": deltaname,
                    "K": K_val,
                    "L": L_val,
                    "t": t,
                    "M": M,
                    "T": T_val,
                    "mean_roughness": mean_roughness,
                    "mean_roughness2": mean_roughness2,
                }
            )

## TimeDep – Delta 0
## TimeDep – Delta atan5
## TimeDep – Delta atan10
## Columnar – Delta 0
## Columnar – Delta atan5
## Columnar – Delta atan10


In [3]:
# Note: we start from scratch the average DataFrame (each time this code is run)
avg_df = pd.DataFrame()
avg_df_path = "../Simulations_2D/average_simulations.pkl"

# Build/update AverageSims dataframe
if summary_rows:
    new_avg = pd.DataFrame(summary_rows).set_index("avg_id")
    if avg_df.empty:
        avg_df = new_avg
    else:
        # ensure we have an index; if not, give it one
        if avg_df.index.name is None:
            avg_df = avg_df.copy()
            avg_df.index.name = "avg_id"
        avg_df = pd.concat([avg_df, new_avg])
        # keep last occurrence per avg_id
        avg_df = avg_df[~avg_df.index.duplicated(keep="last")]

    avg_df.to_pickle(avg_df_path)
    print(f"Saved average simulations ({len(df)}) to:", avg_df_path)

Saved average simulations (393) to: ../Simulations_2D/average_simulations.pkl


### Height-difference correlations

## Inspection of the computed quantities

In [4]:
avg_df_path = "../Simulations_2D/average_simulations.pkl"
avg_df = pd.read_pickle(avg_df_path)

In [5]:
for ki in [1,2,3,5,10]:
    
    df_TimeDep = avg_df[(avg_df["K"] == ki) & (avg_df["T"] == 20000)]

    print(f"TimeDep{ki}", df_TimeDep[["deltaname", "L", "M"]])
    print()

print("###############")
    
for ki in [40,60,80,100]:
    
    df_Columnar = avg_df[(avg_df["K"] == ki) & (avg_df["T"] == 20000)]
    
    print(f"Columnar{ki}", df_Columnar[["deltaname", "L", "M"]])
    print()

TimeDep1                                          deltaname     L    M
avg_id                                                       
TimeDep_delta0_L125_K1.0_T20000_M216             0   125  216
TimeDep_delta0_L250_K1.0_T20000_M94              0   250   94
TimeDep_delta0_L500_K1.0_T20000_M23              0   500   23
TimeDep_delta0_L1000_K1.0_T20000_M1              0  1000    1
TimeDep_deltaatan5_L125_K1.0_T20000_M166     atan5   125  166
TimeDep_deltaatan5_L250_K1.0_T20000_M48      atan5   250   48
TimeDep_deltaatan5_L500_K1.0_T20000_M10      atan5   500   10
TimeDep_deltaatan10_L125_K1.0_T20000_M31    atan10   125   31
TimeDep_deltaatan10_L250_K1.0_T20000_M8     atan10   250    8
TimeDep_deltaatan10_L500_K1.0_T20000_M1     atan10   500    1

TimeDep2                                          deltaname    L    M
avg_id                                                      
TimeDep_delta0_L125_K2.0_T20000_M200             0  125  200
TimeDep_delta0_L250_K2.0_T20000_M53              0  25

In [6]:
avg_df["K"]

avg_id
TimeDep_delta0_L125_K1.0_T20000_M216             1.0
TimeDep_delta0_L250_K1.0_T20000_M94              1.0
TimeDep_delta0_L500_K1.0_T20000_M23              1.0
TimeDep_delta0_L1000_K1.0_T20000_M1              1.0
TimeDep_delta0_L125_K2.0_T20000_M200             2.0
                                               ...  
Columnar_deltaatan5_L250_K100.0_T20000_M6      100.0
Columnar_deltaatan5_L500_K100.0_T20000_M1      100.0
Columnar_deltaatan10_L125_K60.0_T20000_M300     60.0
Columnar_deltaatan10_L250_K60.0_T20000_M75      60.0
Columnar_deltaatan10_L500_K60.0_T20000_M18      60.0
Name: K, Length: 66, dtype: float64

## Plotting results

In [7]:
import matplotlib.pyplot as plt
import scienceplots

plt.style.use(["science","no-latex"])

from plotting_helpers_evolution import plot_roughness_evolution, plot_heightdifference_evolution, plot_structurefactor_evolution, plot_phasecovariance_evolution
from plotting_helpers_collapse import plot_roughness_collapse, plot_heightdifference_collapse, plot_structurefactor_collapse, plot_phasecovariance_collapse

In [8]:
T = 20000
L = 1000